In [1]:
import pandas as pd

splits = {'dev': 'data/secqa_v1_dev.csv', 'val': 'data/secqa_v1_val.csv', 'test': 'data/secqa_v1_test.csv'}
df1 = pd.read_csv("hf://datasets/zefang-liu/secqa/" + splits["dev"])
df2 = pd.read_csv("hf://datasets/zefang-liu/secqa/" + splits["val"])
df3 = pd.read_csv("hf://datasets/zefang-liu/secqa/" + splits["test"])
final_df = pd.concat([df1, df2, df3], axis=0, ignore_index=True)

In [2]:
len(df1), len(df2), len(df3), len(final_df)

(5, 12, 110, 127)

In [3]:
df1

,Question,A,B,C,D,Answer,Explanation
0,What is the purpose of implementing a Guest Wi...,To provide unrestricted access to company reso...,To replace the primary corporate wireless network,To bypass network security protocols,"To offer a separate, secure network for visitors",D,A Guest Wireless Network provides visitors wit...
1,What is a typical indicator that an Intrusion ...,Anomalies or strange behaviors in network traf...,Unauthorized software installation.,Frequent system reboots.,Regular updates to firewall rules.,A,IDS/IPS systems monitor network traffic and ca...
2,What is a security concern associated with Sof...,High costs associated with software licensing.,Consolidation of information with a single pro...,The inability to customize software according ...,The need for constant hardware upgrades.,B,A security concern with SaaS is the consolidat...
3,What is the role of physical controls in a com...,To solely manage digital threats such as virus...,To provide aesthetic enhancements to the secur...,To act as the primary defense against internal...,To protect against physical access and breache...,D,"Physical controls, such as door locks and came..."
4,Which feature should be enabled to hide the na...,SSID Broadcast,Enabling Firewall,MAC Address Filtering,Disabling DHCP,A,Disabling SSID Broadcast hides the network nam...


In [4]:
splits = {'dev': 'data/secqa_v2_dev.csv', 'val': 'data/secqa_v2_val.csv', 'test': 'data/secqa_v2_test.csv'}
df1_1 = pd.read_csv("hf://datasets/zefang-liu/secqa/" + splits["dev"])
df2_1 = pd.read_csv("hf://datasets/zefang-liu/secqa/" + splits["val"])
df3_1 = pd.read_csv("hf://datasets/zefang-liu/secqa/" + splits["test"])
final_df_1 = pd.concat([df1_1, df2_1, df3_1], axis=0, ignore_index=True)

In [5]:
len(df1_1), len(df2_1), len(df3_1), len(final_df_1)

(5, 10, 100, 115)

In [6]:
combined_df = pd.concat([final_df, final_df_1], ignore_index=True)

# 2. Remove duplicates based on the "Question" column
final_df_merged = combined_df.drop_duplicates(subset=["Question"], keep="first")
print(len(final_df_merged))

242


In [7]:
final_df_merged.to_csv("/home/nam/projects/sid/RAG-Assignment3/data/hf_sec_qa/raw/hf_secqa_raw_df.csv")

In [1]:
import pandas as pd
df = pd.read_csv("/home/nam/projects/sid/RAG-Assignment3/data/hf_sec_qa/raw/hf_secqa_raw_df.csv")

In [2]:
import json
from pathlib import Path

output = []

for idx, row in df.iterrows():
    # 1. Cleanly format the options block
    options_text = (
        f"A) {row['A']}\n"
        f"B) {row['B']}\n"
        f"C) {row['C']}\n"
        f"D) {row['D']}"
    )

    # 2. Build the "Super-Chunk"
    # We include the correct answer and explanation to provide full context to the RAG
    combined_text = (
        f"QUESTION: {row['Question']}\n"
        f"ANSWER: {row[row['Answer']]}\n"
        f"nEVIDENCE: {row['Explanation']}"
    ).strip()

    # 3. Create the chunk entry with boilerplate for missing fields
    chunk_entry = {
        "id": int(row.get('Unnamed: 0', idx)), # Use existing ID if available
        "text": combined_text,
        "source": "Multiple Choice Sec QA HF Dataset",  # Boilerplate
        "topic": "Cybersecurity_Quiz",         # Boilerplate
        "target_audience": "Students/Learners", # Boilerplate
        "ground_truth": f"The correct answer to '{row['Question']}' is {row[row['Answer']]}. The reason is as follows: {row['Explanation']}",
        "eval_rubric": "The response must identify the correct multiple-choice option and provide a reasoning consistent with the explanation.",
        "original_evidence": row['Explanation']
    }
    
    output.append(chunk_entry)

# Save to JSON
output_path = Path("/home/nam/projects/sid/RAG-Assignment3/data/hf_sec_qa/processed/hf_secqa_chunks.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2)

print(f"Successfully formatted {len(output)} quiz chunks.")

Successfully formatted 242 quiz chunks.


In [4]:
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

# Load embedding model
model = SentenceTransformer("BAAI/bge-large-en-v1.5")

# Initialize Qdrant
client = QdrantClient(path="/home/nam/projects/sid/RAG-Assignment3/qdrant_data")

COLLECTION_NAME = "hf_secqa_chunks"

# Create collection if not exists
collections = client.get_collections().collections
existing = [c.name for c in collections]

if COLLECTION_NAME not in existing:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=1024,
            distance=Distance.COSINE,
        ),
    )

# Load chunked data
input_path = Path("/home/nam/projects/sid/RAG-Assignment3/data/hf_sec_qa/processed/hf_secqa_chunks.json")

with open(input_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

points = []

for chunk in chunks:
    embedding = model.encode(chunk["text"]).tolist()

    points.append(
        PointStruct(
            id=chunk["id"],
            vector=embedding,
            payload={
                "text": chunk["text"],
                "source": chunk["source"],
                "topic": chunk["topic"],
                # Eval stuff
                "ground_truth": chunk.get("ground_truth"),
                "eval_rubric": chunk.get("eval_rubric"),
                "original_evidence": chunk.get("original_evidence")
            }
        )
    )

# Upload to Qdrant
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

print(f"Inserted {len(points)} chunks into Qdrant")
client.close()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inserted 242 chunks into Qdrant


In [2]:
import requests
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient

OLLAMA_URL = "http://localhost:11434/api/chat"

COLLECTION_NAME = "hf_secqa_chunks"

model = SentenceTransformer("BAAI/bge-large-en-v1.5")

client = QdrantClient(path="/home/nam/projects/sid/RAG-Assignment3/qdrant_data")

query = "What can make an IPS identify as a network attack and explain why you think so"

# Retrieve relevant chunks
query_embedding = model.encode(query).tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding,
    limit=3,
).points

context = "\n\n".join(
    r.payload["text"]
    for r in results
)

print("\nTOP RETRIEVED CHUNKS\n")
for idx, result in enumerate(results, start=1):
    print("=" * 80)
    print(f"Rank #{idx}")
    print(f"Score: {result.score:.4f}")
    print(f"Source: {result.payload['source']}")
    print()
    print(result.payload["text"])
    print()

client.close()

prompt = f"""
You are a cybersecurity assistant.

Use ONLY the provided context to answer the user's question.

If the context does not contain the answer, say:
"I could not find sufficient information."

Provide concise and actionable cybersecurity guidelines.

CONTEXT:
{context}

QUESTION:
{query}
"""

payload = {
    "model": "gemma3:12b-it-qat",
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ],
    "stream": False
}

response = requests.post(OLLAMA_URL, json=payload)

data = response.json()

print("\nGENERATED ANSWER\n")
print(data["message"]["content"])
client.close()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


TOP RETRIEVED CHUNKS

Rank #1
Score: 0.7569
Source: Multiple Choice Sec QA HF Dataset

QUESTION: What is a typical indicator that an Intrusion Detection System (IDS) or Intrusion Prevention System (IPS) might identify as a network attack?
ANSWER: Anomalies or strange behaviors in network traffic.
nEVIDENCE: IDS/IPS systems monitor network traffic and can identify network attacks by detecting anomalies, strange behaviors, or known exploit signatures in the traffic.

Rank #2
Score: 0.7115
Source: Multiple Choice Sec QA HF Dataset

QUESTION: Which indicator would likely suggest an intrusion attempt on a network application?
ANSWER: Alerts from IDS/IPS systems for known exploits, like SQL injection patterns.
nEVIDENCE: Intrusion Detection Systems (IDS) and Intrusion Prevention Systems (IPS) may alert on known exploits, such as SQL injection patterns, indicating an attempted intrusion on the network application.

Rank #3
Score: 0.7101
Source: Multiple Choice Sec QA HF Dataset

QUESTION: In